# ✍️ W2: 상품 카피 & 상세페이지 자동화
**hexa-3 — W02** | ⏱️ 60분 | 🎯 상품 타이틀(40자)+상세설명(200자)+키워드 5개 Gemini 자동생성

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'],capture_output=True)
!pip install -q google-generativeai pandas matplotlib numpy
import matplotlib.pyplot as plt, matplotlib.font_manager as fm, pandas as pd, numpy as np
_n=[f for f in fm.findSystemFonts() if 'Nanum' in f]
if _n: fm.fontManager.addfont(_n[0]); plt.rcParams['font.family']=fm.FontProperties(fname=_n[0]).get_name()
plt.rcParams['axes.unicode_minus']=False
try:
    from google.colab import userdata; API_KEY=userdata.get('GEMINI_API_KEY')
except: API_KEY=input('GEMINI_API_KEY: ')
import google.generativeai as genai; genai.configure(api_key=API_KEY)
model=genai.GenerativeModel('gemini-2.0-flash'); print('✅ 준비 완료')


## Step 1: 쇼핑몰 정보 입력 (✏️)

In [ ]:
STORE={'name':'✏️ [쇼핑몰명]','platform':'✏️ [스마트스토어|쿠팡|자사몰]'}
print('✅',STORE['name'])


## Step 2: 데이터 로드

In [ ]:
import io, requests
URL='https://raw.githubusercontent.com/Reasonofmoon/hexa-3/main/shared/retail_products_sample.csv'
try:
    df=pd.read_csv(URL,encoding='utf-8-sig'); print(f'✅ GitHub 로드: {len(df)}행')
except:
    try:
        from google.colab import files; up=files.upload()
        if up: df=pd.read_csv(io.BytesIO(list(up.values())[0]),encoding='utf-8-sig')
        else: raise Exception('취소됨')
    except:
        df=pd.DataFrame({'상품명':['무선이어폰','스마트워치','USB허브','블루투스스피커','충전기'],'카테고리':['전자기기']*5,'가격':[45000,120000,25000,89000,15000],'재고':[50,20,100,30,200],'월판매량':[150,80,200,60,300],'평점':[4.5,4.8,4.2,4.6,4.1],'리뷰수':[230,150,180,90,410]})
        print('📋 기본 샘플 사용')
print(df.head())


## Step 3: 상품별 카피 자동 생성

In [ ]:
results=[]
for _,row in df.head(5).iterrows():
    p=f"""스마트스토어 상품 카피를 작성하세요.
상품:{row['상품명']} ({row['가격']:,}원)
1. 검색 타이틀 (40자 이내, 핵심 키워드 포함)
2. 상세 설명 (200자, 특징 3가지)
3. 검색 키워드 5개 (쉼표 구분)"""
    resp=model.generate_content(p).text
    results.append({'상품명':row['상품명'],'카피':resp[:300]})
    print(f'[{row["상품명"]}]\n{resp[:200]}\n')


## Step 4: ZIP 다운로드

In [ ]:
import zipfile
for r in results:
    with open(f'{r["상품명"]}.txt','w',encoding='utf-8') as f: f.write(r['카피'])
with zipfile.ZipFile('product_copy.zip','w') as z:
    for r in results: z.write(f'{r["상품명"]}.txt')
from google.colab import files; files.download('product_copy.zip')
print('✅ 완료!')
